# classifai — Quick Start

Classify any text dataset in minutes using AI or traditional clustering.

This notebook walks you through:
1. Loading a sample support ticket dataset
2. Classifying tickets by department using an LLM
3. Discovering hidden patterns with unsupervised clustering
4. Visualizing and evaluating the results

**Two ways to run the LLM backend:**
- **OpenAI** (API key required, ~$0.01 for this demo)
- **Ollama** (free, runs locally — [install here](https://ollama.com))

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JuanLara18/classifai/blob/main/notebooks/quickstart.ipynb)


## 0. Setup

In [ ]:
# Install dependencies
# If running in Colab, uncomment the next block:

# !git clone https://github.com/JuanLara18/classifai.git
# %cd classifai
# !pip install -q openai instructor pydantic pandas plotly scikit-learn umap-learn hdbscan sentence-transformers

In [ ]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from pathlib import Path

# classifai imports
import sys
sys.path.insert(0, str(Path.cwd().parent))  # adjust if running from a different directory

from classifai.backends import LLMBackend

print('Setup complete.')

## 1. Load the sample dataset

56 realistic support tickets across 6 departments, with ground-truth labels
so we can measure accuracy after classification.

In [ ]:
df = pd.read_csv('../data/support_tickets_sample.csv')
print(f'{len(df)} tickets loaded')
df.head(3)

In [ ]:
# Class distribution
fig = px.bar(
    df['true_category'].value_counts().reset_index(),
    x='true_category', y='count',
    title='Ticket distribution by department (ground truth)',
    labels={'true_category': 'Department', 'count': 'Tickets'},
    color='true_category',
    color_discrete_sequence=px.colors.qualitative.Set2,
)
fig.update_layout(showlegend=False, xaxis_tickangle=-20)
fig.show()

## 2. Classify with an LLM

Choose your provider below. The backend guarantees a valid label for every
row — no regex, no post-processing.

In [ ]:
# ── Provider choice ────────────────────────────────────────────────────────
# Option A: OpenAI (set your key below or as an env variable)
import os
# os.environ['OPENAI_API_KEY'] = 'sk-...'   # uncomment and fill in

PROVIDER = 'openai'   # change to 'ollama' if you have Ollama running locally
MODEL    = 'gpt-4o-mini'   # for Ollama: 'llama3', 'mistral', 'phi4', etc.
# ──────────────────────────────────────────────────────────────────────────

CATEGORIES = [
    'Billing & Payments',
    'Technical Support',
    'Account & Access',
    'Shipping & Delivery',
    'Returns & Refunds',
    'Product Information',
]

clf = LLMBackend(
    categories=CATEGORIES,
    model=MODEL,
    provider=PROVIDER,
    temperature=0.0,
    batch_size=20,
    max_workers=4,
)

print(f'Backend ready: {PROVIDER} / {MODEL}')

In [ ]:
# Combine subject + body for richer context
text_col = df['subject'] + '\n\n' + df['body']

# Classify — unique-value optimization reduces API calls automatically
df['predicted_category'] = clf.predict(text_col)

print('Classification complete.')
print(f"API calls made: {clf.stats['api_calls']} (unique-value optimization active)")
df[['id', 'subject', 'true_category', 'predicted_category']].head(8)

## 3. Evaluate accuracy

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np

accuracy = (df['predicted_category'] == df['true_category']).mean()
print(f'Overall accuracy: {accuracy:.1%}\n')
print(classification_report(df['true_category'], df['predicted_category']))

In [ ]:
# Confusion matrix
labels = sorted(CATEGORIES)
cm = confusion_matrix(df['true_category'], df['predicted_category'], labels=labels)

fig = px.imshow(
    cm,
    x=labels, y=labels,
    color_continuous_scale='Blues',
    title='Confusion matrix — LLM classification',
    labels={'x': 'Predicted', 'y': 'Actual', 'color': 'Count'},
    text_auto=True,
)
fig.update_xaxes(tickangle=-30)
fig.show()

## 4. Discover patterns with clustering

What if you don't know the categories in advance?  
Clustering finds natural groups in your data without any labels.

In [ ]:
from sentence_transformers import SentenceTransformer
import umap
import hdbscan

# 1. Embed text with a sentence transformer
encoder = SentenceTransformer('all-MiniLM-L6-v2')
embeddings = encoder.encode(text_col.tolist(), show_progress_bar=True)
print(f'Embeddings shape: {embeddings.shape}')

In [ ]:
# 2. Reduce to 2D with UMAP (for visualization) and 10D (for clustering)
reducer_2d  = umap.UMAP(n_components=2,  random_state=42, min_dist=0.1)
reducer_10d = umap.UMAP(n_components=10, random_state=42)

emb_2d  = reducer_2d.fit_transform(embeddings)
emb_10d = reducer_10d.fit_transform(embeddings)

# 3. Cluster with HDBSCAN
clusterer = hdbscan.HDBSCAN(min_cluster_size=4, min_samples=2)
cluster_labels = clusterer.fit_predict(emb_10d)

n_clusters = len(set(cluster_labels)) - (1 if -1 in cluster_labels else 0)
n_noise    = (cluster_labels == -1).sum()
print(f'Found {n_clusters} clusters, {n_noise} noise points')

In [ ]:
# 4. Visualize
plot_df = pd.DataFrame({
    'x': emb_2d[:, 0],
    'y': emb_2d[:, 1],
    'cluster': cluster_labels.astype(str),
    'true_category': df['true_category'].values,
    'subject': df['subject'].values,
})

fig = px.scatter(
    plot_df,
    x='x', y='y',
    color='true_category',
    symbol='cluster',
    hover_data=['subject', 'cluster'],
    title='UMAP embedding — colored by true category, shape by HDBSCAN cluster',
    color_discrete_sequence=px.colors.qualitative.Set2,
    width=800, height=550,
)
fig.update_traces(marker=dict(size=9, opacity=0.85))
fig.show()

## 5. Side-by-side comparison

When you run both approaches, you can compare them directly.

In [ ]:
df['cluster'] = [f'Cluster {c}' if c != -1 else 'Noise' for c in cluster_labels]

fig = px.scatter(
    plot_df,
    x='x', y='y',
    color='cluster',
    hover_data=['subject', 'true_category'],
    title='HDBSCAN clusters in embedding space',
    color_discrete_sequence=px.colors.qualitative.Pastel,
    width=800, height=550,
)
fig.update_traces(marker=dict(size=9, opacity=0.85))
fig.show()

In [ ]:
# Summary table
summary = df[['id', 'subject', 'true_category', 'predicted_category', 'cluster']].copy()
summary['correct'] = summary['true_category'] == summary['predicted_category']
summary.style.applymap(lambda v: 'background-color: #d4edda' if v else 'background-color: #f8d7da',
                        subset=['correct'])

## Next steps

- **Scale up**: run `python main.py --config config.example.yaml` on your own CSV
- **Custom prompt**: add `prompt_template` to `LLMBackend` to guide the LLM
- **Free & local**: switch `provider='ollama'` and `model='llama3'` — no API costs
- **Multi-label**: define multiple `LLMBackend` instances for different classification axes
- **Full pipeline**: use the YAML config for checkpointing, HTML reports, and cost tracking

See the [README](../README.md) and [`config.example.yaml`](../config.example.yaml) for details.